# RAG with LangChain
## Week 8 — Day 3 | DI GenAI & Machine Learning Bootcamp 2026

**Goal:** Build a Retrieval-Augmented Generation (RAG) question-answering system using:
- **LangChain** as the orchestration framework
- **`m-ric/huggingface_doc`** as the knowledge base
- **`google/flan-t5-small`** as the local language model (no API key required)
- **FAISS** as the vector store
- **`sentence-transformers/all-MiniLM-L6-v2`** for embeddings

```
User Question
      ↓
Embed query (all-MiniLM-L6-v2)
      ↓
FAISS similarity search → top-k chunks
      ↓
Prompt = chunks + question
      ↓
flan-t5-small generates grounded answer
```

## Task 1 — Setup and Imports

In [ ]:
# Install all required dependencies
%pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    transformers \
    datasets \
    torch \
    huggingface_hub

print("Dependencies installed ✓")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Dataset loading
from datasets import load_dataset

# LangChain — document schema
from langchain.schema import Document

# LangChain — text splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

# LangChain — embeddings via Sentence Transformers
from langchain_huggingface import HuggingFaceEmbeddings

# LangChain — FAISS vector store
from langchain_community.vectorstores import FAISS

# LangChain — LLM wrapper for Hugging Face pipelines
from langchain_huggingface import HuggingFacePipeline

# LangChain — RetrievalQA chain
from langchain.chains import RetrievalQA

# Hugging Face transformers pipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("All imports OK ✓")

## Task 2 — Load the Dataset

In [ ]:
# Load the first 200 rows of the HuggingFace documentation dataset
print("Loading m-ric/huggingface_doc (train[:200])…")
dataset = load_dataset("m-ric/huggingface_doc", split="train[:200]")

print(f"\nDataset loaded ✓")
print(f"  Rows    : {len(dataset)}")
print(f"  Columns : {dataset.column_names}")

In [ ]:
# Preview one example row
print("=== Example row ===\n")
row = dataset[0]
for col, val in row.items():
    preview = str(val)[:200] + "…" if len(str(val)) > 200 else str(val)
    print(f"[{col}]\n  {preview}\n")

## Task 3 — Convert Rows into LangChain Documents

LangChain expects `Document` objects with:
- `page_content` — the main text the retriever embeds and searches
- `metadata` — a dict with provenance info (e.g. `source`)

In [ ]:
# Convert each dataset row into a LangChain Document
docs = [
    Document(
        page_content = row["text"],
        metadata     = {"source": row["source"]},
    )
    for row in dataset
    if row["text"] and row["text"].strip()   # skip empty texts
]

print(f"LangChain Documents created: {len(docs)}")
print(f"\n--- Document 0 ---")
print(f"page_content : {docs[0].page_content[:300]}…")
print(f"metadata     : {docs[0].metadata}")
print(f"\n--- Document 1 ---")
print(f"page_content : {docs[1].page_content[:300]}…")
print(f"metadata     : {docs[1].metadata}")

## Task 4 — Chunk the Documents

Why chunk?
- Embedding models have a maximum input length (e.g. 256 tokens for `all-MiniLM-L6-v2`).
- Smaller, focused chunks lead to more precise similarity matches.
- `chunk_overlap` ensures a sentence split at a boundary isn't lost.

We compare **two chunking strategies** to observe the effect on retrieved content.

In [ ]:
# Strategy A — small chunks (fine-grained, high precision)
splitter_A = RecursiveCharacterTextSplitter(
    chunk_size    = 256,
    chunk_overlap = 32,
    length_function = len,
    separators      = ["\n\n", "\n", ". ", " ", ""],
)
chunks_A = splitter_A.split_documents(docs)

# Strategy B — larger chunks (more context per chunk, lower precision)
splitter_B = RecursiveCharacterTextSplitter(
    chunk_size    = 512,
    chunk_overlap = 64,
    length_function = len,
    separators      = ["\n\n", "\n", ". ", " ", ""],
)
chunks_B = splitter_B.split_documents(docs)

print(f"Strategy A (size=256, overlap=32 ) → {len(chunks_A):>5} chunks")
print(f"Strategy B (size=512, overlap=64 ) → {len(chunks_B):>5} chunks")
print()
print(f"Avg chunk length — A: {sum(len(c.page_content) for c in chunks_A)//len(chunks_A)} chars")
print(f"Avg chunk length — B: {sum(len(c.page_content) for c in chunks_B)//len(chunks_B)} chars")
print()
print(f"--- Sample chunk (Strategy A) ---")
print(chunks_A[3].page_content)
print(f"\nmetadata: {chunks_A[3].metadata}")

In [ ]:
# Observation: how chunking strategy affects content
sample_doc_idx = 5
print(f"=== Original document {sample_doc_idx} (first 500 chars) ===")
print(docs[sample_doc_idx].page_content[:500])

# Find all chunks that came from this document
src = docs[sample_doc_idx].metadata["source"]
chunks_a_from_doc = [c for c in chunks_A if c.metadata["source"] == src]
chunks_b_from_doc = [c for c in chunks_B if c.metadata["source"] == src]

print(f"\nStrategy A splits doc into {len(chunks_a_from_doc)} chunk(s)")
print(f"Strategy B splits doc into {len(chunks_b_from_doc)} chunk(s)")
print()
print("Insight: Larger chunks (B) preserve more context per retrieval hit.")
print("Smaller chunks (A) are more targeted but may cut off context mid-sentence.")

We'll use **Strategy A (size=256)** for the rest of the notebook — its smaller chunks match well with the 256-token limit of `all-MiniLM-L6-v2`.

## Task 5 — Build the Vector Store and Retriever

In [ ]:
# Initialize the embedding model
print("Loading sentence-transformers/all-MiniLM-L6-v2…")
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device": "cpu"},
)
print(f"Embedding model loaded ✓")

# Verify with a test sentence
test_vec = embedding_model.embed_query("What is a Hugging Face pipeline?")
print(f"Embedding dimension : {len(test_vec)}")

In [ ]:
# Build FAISS vector store from all Strategy-A chunks
print(f"Building FAISS index from {len(chunks_A)} chunks (this takes ~1–2 min)…")
vectorstore = FAISS.from_documents(
    documents = chunks_A,
    embedding = embedding_model,
)
print(f"FAISS vector store built ✓")
print(f"  Vectors indexed : {vectorstore.index.ntotal}")

In [ ]:
# Build retrievers with different k values to compare coverage vs. precision
retriever_k2 = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever_k4 = vectorstore.as_retriever(search_kwargs={"k": 4})  # default
retriever_k6 = vectorstore.as_retriever(search_kwargs={"k": 6})

print("Retrievers created:")
print("  retriever_k2 → returns top 2 chunks (high precision, low coverage)")
print("  retriever_k4 → returns top 4 chunks (balanced)")
print("  retriever_k6 → returns top 6 chunks (high coverage, lower precision)")

## Task 6 — Retrieval Sanity Check (Required)

Before generating answers we must verify the retriever actually returns relevant chunks.  
If chunks are off-topic, we adjust chunking parameters or k.

In [ ]:
def show_retrieved_chunks(question: str, retriever, label: str = ""):
    """Retrieve and display chunks with their sources."""
    retrieved = retriever.invoke(question)
    print(f"{'='*65}")
    print(f"Question ({label}): {question}")
    print(f"Chunks returned : {len(retrieved)}")
    print("=" * 65)
    for i, chunk in enumerate(retrieved, 1):
        print(f"\n[Chunk {i}]  source: {chunk.metadata.get('source', 'unknown')}")
        print(f"{chunk.page_content[:300]}…")
    print()


# Sanity check — a question clearly answerable from the HuggingFace docs
test_question = "What is a Hugging Face pipeline and how do I use it?"

print("--- k=2 ---")
show_retrieved_chunks(test_question, retriever_k2, label="k=2")

print("--- k=4 ---")
show_retrieved_chunks(test_question, retriever_k4, label="k=4")

print("--- k=6 ---")
show_retrieved_chunks(test_question, retriever_k6, label="k=6")

In [ ]:
# Second sanity check with a different question
test_question_2 = "How do I fine-tune a model with the Trainer API?"
show_retrieved_chunks(test_question_2, retriever_k4, label="k=4")

print("Retrieval check summary:")
print("  k=2 : Fewer chunks — very precise but may miss context")
print("  k=4 : Good balance — recommended for this dataset size")
print("  k=6 : More coverage — useful when the answer spans multiple sections")

**Retrieval looks good ✓** — the returned chunks contain HuggingFace documentation text directly relevant to each question. We'll proceed with `k=4` as the default retriever.

## Task 7 — RAG Question Answering with Flan-T5

### 7.1 — Load `google/flan-t5-small`

In [ ]:
print("Loading google/flan-t5-small…")
model_id  = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model  = AutoModelForSeq2SeqLM.from_pretrained(model_id)

n_params = sum(p.numel() for p in lm_model.parameters())
print(f"Model loaded ✓  ({n_params/1e6:.0f}M parameters)")

# Build a text2text generation pipeline
hf_pipe = pipeline(
    "text2text-generation",
    model         = lm_model,
    tokenizer     = tokenizer,
    max_new_tokens = 256,
    do_sample     = False,   # greedy decoding for deterministic answers
)

# Wrap in LangChain
llm = HuggingFacePipeline(pipeline=hf_pipe)
print("LangChain LLM wrapper created ✓")

### 7.2 — Build the RetrievalQA Chain

In [ ]:
# Assemble the RAG chain
# chain_type="stuff" → concatenates all retrieved chunks into a single prompt
rag_chain = RetrievalQA.from_chain_type(
    llm             = llm,
    chain_type      = "stuff",
    retriever       = retriever_k4,
    return_source_documents = True,   # return chunks used for the answer
)

print("RetrievalQA chain assembled ✓")
print("  chain_type : stuff  (retrieved chunks stuffed into a single prompt)")
print("  retriever  : k=4")
print("  LLM        : google/flan-t5-small")

### 7.3 — Ask Questions and Inspect Sources

In [ ]:
def ask_rag(question: str, chain) -> None:
    """
    Run the RAG chain, print the generated answer, and show the sources
    of the retrieved chunks so we can verify the model is grounded in the dataset.
    """
    result   = chain.invoke({"query": question})
    answer   = result["result"]
    sources  = result["source_documents"]

    print(f"{'='*65}")
    print(f"Question: {question}")
    print(f"{'='*65}")
    print(f"Answer  : {answer}")
    print(f"\nSources ({len(sources)} retrieved chunks):")
    for i, doc in enumerate(sources, 1):
        print(f"  [{i}] source: {doc.metadata.get('source', 'unknown')}")
        print(f"       {doc.page_content[:150]}…")
    print()


print("ask_rag() helper defined ✓")

In [ ]:
# Question 1 — fundamental concept
ask_rag("What is a Hugging Face pipeline?", rag_chain)

In [ ]:
# Question 2 — training API
ask_rag("How do I fine-tune a model using the Trainer API?", rag_chain)

In [ ]:
# Question 3 — tokenizers
ask_rag("What is a tokenizer and why is it needed?", rag_chain)

In [ ]:
# Question 4 — datasets library
ask_rag("How do I load a dataset from the Hugging Face Hub?", rag_chain)

In [ ]:
# Question 5 — model sharing / hub
ask_rag("How do I share a model on the Hugging Face Hub?", rag_chain)

### 7.4 — Compare k values on the same question

In [ ]:
comparison_question = "What is a Hugging Face pipeline?"

for k, retriever in [(2, retriever_k2), (4, retriever_k4), (6, retriever_k6)]:
    chain_k = RetrievalQA.from_chain_type(
        llm             = llm,
        chain_type      = "stuff",
        retriever       = retriever,
        return_source_documents = True,
    )
    result = chain_k.invoke({"query": comparison_question})
    print(f"--- k={k} ---")
    print(f"Answer  : {result['result']}")
    print(f"Sources : {[d.metadata.get('source','?') for d in result['source_documents']]}")
    print()

### 7.5 — Debugging: when retrieval goes wrong

In [ ]:
# Ask a question the dataset does NOT contain → model should admit uncertainty
out_of_scope_q = "What are the best restaurants in Paris?"

print("=== Out-of-scope question (not in HF docs) ===")
print(f"Question: {out_of_scope_q}")

# Inspect what chunks are retrieved (they'll be irrelevant)
chunks = retriever_k4.invoke(out_of_scope_q)
print(f"\nRetrieved chunks (should be irrelevant):")
for i, c in enumerate(chunks, 1):
    print(f"  [{i}] {c.page_content[:120]}…")

print()
result = rag_chain.invoke({"query": out_of_scope_q})
print(f"Answer  : {result['result']}")
print()
print("Observation: The retrieved chunks are from HF docs (irrelevant to the question).")
print("The answer may be incorrect/hallucinated — this is the key weakness of RAG")
print("when the knowledge base does not cover the query.")

## Summary & Key Takeaways

| Component | Choice | Why |
|---|---|---|
| Dataset | `m-ric/huggingface_doc` train[:200] | Small, fast, text + source columns |
| Document schema | `langchain.schema.Document` | Pairs `page_content` with `metadata` |
| Chunking | `RecursiveCharacterTextSplitter` | Respects sentence/paragraph boundaries |
| Embeddings | `all-MiniLM-L6-v2` (384-dim) | Small, fast, good semantic quality |
| Vector store | FAISS | Local, no API key, exact inner-product |
| LLM | `google/flan-t5-small` | No key, seq2seq, instruction-following |
| Chain | `RetrievalQA` (stuff) | Simplest chain: stuff chunks into prompt |

### Chunking effect

| Strategy | chunk_size | overlap | Chunks | Effect |
|---|---|---|---|---|
| A (used) | 256 chars | 32 | more | Precise, fits embedding model limit |
| B | 512 chars | 64 | fewer | More context, but can exceed token limit |

### k effect (retrieval)

| k | Chunks in prompt | Precision | Coverage |
|---|---|---|---|
| 2 | few | high | low — may miss answer |
| 4 | balanced | good | good |
| 6 | many | lower | high — may confuse LLM |

### When RAG fails
1. **Out-of-scope query** — retrieved chunks are irrelevant; LLM hallucinates from wrong context.
2. **Chunks too large** — exceed LLM context window; answer cut off or corrupted.
3. **k too small** — answer requires multiple passages; insufficient context.
4. **Poor embedding alignment** — query phrasing doesn't match document language; tune chunking or try a domain-specific embedding model.